In [2]:
# ============================================================
# STEP 3 — DATA CLEANING & PREPROCESSING
# ============================================================

# Import all the libraries we need
import pandas as pd                                  # for working with tables
import numpy as np                                   # for math operations
from sklearn.preprocessing import LabelEncoder       # for label encoding
from sklearn.preprocessing import StandardScaler     # for feature scaling

# Load the dataset fresh
df = pd.read_csv("HR_Job_Placement_Dataset.csv")
print("Original shape:", df.shape)                   # (51500, 26)


Original shape: (51500, 26)


In [3]:
# ============================================================
# 🧹 CLEANING STEP 1: Fix Noisy Text Values
# ============================================================
# Problem: 'Male', 'male', 'MALE' all mean the same thing
#          but Python treats them as 3 different words!
# Fix   : Strip spaces + convert everything to Title Case
#         'male' → 'Male' | ' no ' → 'No' | 'YES' → 'Yes'

# First, identify all text (string) columns
str_cols = [col for col in df.columns
            if df[col].dtype == 'object'             # 'object' = text column
            or str(df[col].dtype) == 'str']          # pandas 3.x string type

for col in str_cols:                                 # loop through each text column
    df[col] = df[col].astype(str)                    # convert to standard Python string
    df[col] = df[col].str.strip()                    # remove spaces from both ends
    df[col] = df[col].str.title()                    # Title Case: 'male' → 'Male'
    df[col] = df[col].replace('Nan', np.nan) 

In [4]:
# Verify the fix
print("\nGender values after fix:", df['gender'].unique())
# Expected: ['Male' 'Female'] — no more 'male' or 'female'!

print("Company tier after fix:", df['company_tier'].unique())
# Expected: ['Tier 1' 'Tier 2' 'Tier 3'] — no trailing spaces!


Gender values after fix: ['Male' 'Female']
Company tier after fix: ['Tier 3' 'Tier 1' 'Tier 2']


In [5]:
# ============================================================
# 🧹 CLEANING STEP 2: Remove Duplicate Rows
# ============================================================
# Problem: 1,376 rows are exact copies of other rows
# Fix   : Keep only the first occurrence, delete the rest
# NOTE  : We do this AFTER fixing text — because 'male' and
#         'Male' in the same row now look identical, revealing
#         even more duplicates than we first found!

before_dedup = len(df)                               # count rows before
df = df.drop_duplicates()                            # remove duplicate rows
after_dedup = len(df)                                # count rows after

print(f"\nDuplicates removed: {before_dedup - after_dedup}")
print(f"Shape after removing duplicates: {df.shape}")
# After fixing case + removing duplicates: 51500 → 50000 rows



Duplicates removed: 1500
Shape after removing duplicates: (50000, 26)


In [6]:
df.isnull().sum()

age_years                       0
gender                          0
ssc_percentage               2500
hsc_percentage               2000
degree_percentage               0
degree_specialization           0
technical_score                 0
aptitude_score                  0
communication_score             0
skills_match_percentage         0
certifications_count            0
internship_experience           0
years_of_experience             0
career_switch_willingness    8224
relevant_experience          8183
previous_ctc_lpa                0
expected_ctc_lpa                0
company_tier                    0
job_role_match               5949
competition_level               0
bond_requirement                0
notice_period_days           1532
layoff_history               8234
employment_gap_months        1071
relocation_willingness       8227
status                          0
dtype: int64

In [7]:
# ============================================================
# 🧹 CLEANING STEP 3: Handle Missing Values
# ============================================================


print("\nSkewness of numeric columns with missing values:")
print(f"  ssc_percentage     : {df['ssc_percentage'].skew():.2f}  → use MEAN")
print(f"  hsc_percentage     : {df['hsc_percentage'].skew():.2f}  → use MEAN")
print(f"  notice_period_days : {df['notice_period_days'].skew():.2f}  → use MEDIAN")
print(f"  employment_gap_months: {df['employment_gap_months'].skew():.2f} → use MEDIAN")

# Fill ssc_percentage with MEAN (skew ≈ 0, no outliers)
df['ssc_percentage'] = df['ssc_percentage'].fillna(
    df['ssc_percentage'].mean()
)

# Fill hsc_percentage with MEAN (skew ≈ 0, no outliers)
df['hsc_percentage'] = df['hsc_percentage'].fillna(
    df['hsc_percentage'].mean()
)

# Fill notice_period_days with MEDIAN (skew > 1, has outliers)
df['notice_period_days'] = df['notice_period_days'].fillna(
    df['notice_period_days'].median()
)

# Fill employment_gap_months with MEDIAN (skew > 1, has outliers)
df['employment_gap_months'] = df['employment_gap_months'].fillna(
    df['employment_gap_months'].median()
)


Skewness of numeric columns with missing values:
  ssc_percentage     : 0.05  → use MEAN
  hsc_percentage     : 0.01  → use MEAN
  notice_period_days : 1.35  → use MEDIAN
  employment_gap_months: 1.72 → use MEDIAN


In [8]:
df.isnull().sum()

age_years                       0
gender                          0
ssc_percentage                  0
hsc_percentage                  0
degree_percentage               0
degree_specialization           0
technical_score                 0
aptitude_score                  0
communication_score             0
skills_match_percentage         0
certifications_count            0
internship_experience           0
years_of_experience             0
career_switch_willingness    8224
relevant_experience          8183
previous_ctc_lpa                0
expected_ctc_lpa                0
company_tier                    0
job_role_match               5949
competition_level               0
bond_requirement                0
notice_period_days              0
layoff_history               8234
employment_gap_months           0
relocation_willingness       8227
status                          0
dtype: int64

In [9]:
# ── WHY MODE for categorical? ────────────────────────────
# Mode = most common value


cat_missing_cols = [
    'career_switch_willingness',    # 16.4% missing
    'relevant_experience',          # 16.4% missing
    'job_role_match',               # 11.9% missing
    'layoff_history',               # 16.4% missing
    'relocation_willingness'        # 16.5% missing
]

for col in cat_missing_cols:
    mode_value = df[col].mode()[0]              # find most common value
    df[col] = df[col].fillna(mode_value)        # fill blanks with it
    print(f"  {col}: filled with '{mode_value}'")

# ✅ Verify — should show 0 missing values everywhere
print(f"\nTotal missing values remaining: {df.isnull().sum().sum()}")
# Expected: 0 ✅

  career_switch_willingness: filled with 'Not Willing'
  relevant_experience: filled with 'Relevant'
  job_role_match: filled with 'Matched'
  layoff_history: filled with 'No'
  relocation_willingness: filled with 'Willing'

Total missing values remaining: 0


In [10]:
df.iloc[0] # Check the data types and missing values after cleaning

age_years                                  27
gender                                   Male
ssc_percentage                      65.061656
hsc_percentage                      83.842578
degree_percentage                   75.856526
degree_specialization        Computer Science
technical_score                     58.221909
aptitude_score                      89.566305
communication_score                 64.474484
skills_match_percentage             79.548913
certifications_count                        2
internship_experience                      No
years_of_experience                         1
career_switch_willingness             Willing
relevant_experience                  Relevant
previous_ctc_lpa                     3.530557
expected_ctc_lpa                      5.80585
company_tier                           Tier 3
job_role_match                    Not Matched
competition_level                      Medium
bond_requirement                 Not Required
notice_period_days                

In [11]:
# ── CLEANING STEP 1: Fix Noisy Text Values ──────────────────
str_cols = [col for col in df.columns
            if str(df[col].dtype) in ['object', 'str']]

for col in str_cols:
    df[col] = df[col].astype(str)       # convert to string
    df[col] = df[col].str.strip()       # ← THIS removes "Tier 1 " spaces
    df[col] = df[col].str.title()       # ← THIS fixes male → Male
    df[col] = df[col].replace('Nan', np.nan)


In [12]:
# ── FINAL VERIFICATION after all cleaning ───────────────────

# Check 1: company_tier should have exactly 3 unique values
print("Company Tier unique:", df['company_tier'].unique())
# ✅ Expected: ['Tier 1' 'Tier 2' 'Tier 3']

# Check 2: gender should have exactly 2 unique values
print("Gender unique:", df['gender'].unique())
# ✅ Expected: ['Male' 'Female']

# Check 3: internship should have exactly 2 unique values
print("Internship unique:", df['internship_experience'].unique())
# ✅ Expected: ['No' 'Yes']

# Check 4: total shape
print("Final shape:", df.shape)
# ✅ Expected: (50000, 26)
## 🗺️ The Golden Rule

Company Tier unique: ['Tier 3' 'Tier 1' 'Tier 2']
Gender unique: ['Male' 'Female']
Internship unique: ['No' 'Yes']
Final shape: (50000, 26)


In [13]:
# ✅ SAVE HERE — before encoding (for EDA use)
df.to_csv("HR_Cleaned_For_EDA.csv", index=False)
print("EDA file saved! Shape:", df.shape)  # (50000, 26)


EDA file saved! Shape: (50000, 26)


In [14]:
# ============================================================
# 🔢 ENCODING STEP 1: Encode the Target Variable
# ============================================================

df['status_encoded'] = df['status'].map({
    'Placed'    : 1,                            # map 'Placed' to 1
    'Not Placed': 0                             # map 'Not Placed' to 0
})

print("\nTarget variable encoding check:")
print(df[['status', 'status_encoded']].value_counts())



Target variable encoding check:
status      status_encoded
Not Placed  0                 34870
Placed      1                 15130
Name: count, dtype: int64


In [15]:
# ============================================================
# 🔢 ENCODING STEP 1: Encode the Target Variable
# ============================================================

df['status_encoded'] = df['status'].map({
    'Placed'    : 1,                            # map 'Placed' to 1
    'Not Placed': 0                             # map 'Not Placed' to 0
})

print("\nTarget variable encoding check:")
print(df[['status', 'status_encoded']].value_counts())


# ============================================================
# 🔢 ENCODING STEP 2: Label Encoding (Binary Columns)
# ============================================================


le = LabelEncoder()                             # create the encoder tool

binary_cols = [
    'internship_experience',        # Yes/No → 1/0
    'career_switch_willingness',    # Willing/Not Willing → 1/0
    'relevant_experience',          # Relevant/Not Relevant → 1/0
    'job_role_match',               # Matched/Not Matched → 1/0
    'bond_requirement',             # Required/Not Required → 1/0
    'layoff_history',               # Yes/No → 1/0
    'relocation_willingness'        # Willing/Not Willing → 1/0
]

for col in binary_cols:
    # fit_transform: learns the mapping AND applies it
    df[col + '_encoded'] = le.fit_transform(df[col])

    # Show what mapping was used
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f"  {col}: {mapping}")


Target variable encoding check:
status      status_encoded
Not Placed  0                 34870
Placed      1                 15130
Name: count, dtype: int64
  internship_experience: {'No': np.int64(0), 'Yes': np.int64(1)}
  career_switch_willingness: {'Not Willing': np.int64(0), 'Willing': np.int64(1)}
  relevant_experience: {'Not Relevant': np.int64(0), 'Relevant': np.int64(1)}
  job_role_match: {'Matched': np.int64(0), 'Not Matched': np.int64(1)}
  bond_requirement: {'Not Required': np.int64(0), 'Required': np.int64(1)}
  layoff_history: {'No': np.int64(0), 'Yes': np.int64(1)}
  relocation_willingness: {'Not Willing': np.int64(0), 'Willing': np.int64(1)}


In [16]:
df.head()

,age_years,gender,ssc_percentage,hsc_percentage,degree_percentage,degree_specialization,technical_score,aptitude_score,communication_score,skills_match_percentage,...,relocation_willingness,status,status_encoded,internship_experience_encoded,career_switch_willingness_encoded,relevant_experience_encoded,job_role_match_encoded,bond_requirement_encoded,layoff_history_encoded,relocation_willingness_encoded
0,27,Male,65.061656,83.842578,75.856526,Computer Science,58.221909,89.566305,64.474484,79.548913,...,Not Willing,Not Placed,0,0,1,1,1,0,0,0
1,24,Male,67.885626,64.973305,73.093588,Electronics,71.927978,54.591971,61.077306,73.316134,...,Not Willing,Not Placed,0,1,0,1,0,1,0,0
2,33,Female,73.892471,68.834121,90.196460,Information Technology,72.445041,58.587088,79.494739,75.466980,...,Not Willing,Placed,1,1,1,1,1,0,0,0
3,31,Male,74.145568,76.255126,75.586731,Mechanical,78.855676,61.022065,53.740386,73.676449,...,Willing,Not Placed,0,0,0,1,0,0,1,1
4,28,Male,60.475937,65.786336,80.801010,Information Technology,68.286776,65.713731,61.438314,88.994847,...,Willing,Not Placed,0,0,1,0,0,0,0,1


In [17]:
# ============================================================
# 🔢 ENCODING STEP 3: One-Hot Encoding (Multi-Category Columns)
# ============================================================

multi_cols = [
    'gender',                   # Male, Female
    'degree_specialization',    # CS, Electronics, IT, Mech, Others
    'company_tier',             # Tier 1, Tier 2, Tier 3
    'competition_level'         # Low, Medium, High
]

# pd.get_dummies creates the new 0/1 columns automatically
# drop_first=False → keep ALL categories (safer for understanding)
# ✅ NEW — forces proper 1s and 0s always
df = pd.get_dummies(df, columns=multi_cols, drop_first=False, dtype=int)

print(f"\nShape after One-Hot Encoding: {df.shape}")
# More columns now because we created new ones for each category!



Shape after One-Hot Encoding: (50000, 43)


In [18]:
# Shows ALL column names exactly as they are
print(df.columns.tolist())

['age_years', 'ssc_percentage', 'hsc_percentage', 'degree_percentage', 'technical_score', 'aptitude_score', 'communication_score', 'skills_match_percentage', 'certifications_count', 'internship_experience', 'years_of_experience', 'career_switch_willingness', 'relevant_experience', 'previous_ctc_lpa', 'expected_ctc_lpa', 'job_role_match', 'bond_requirement', 'notice_period_days', 'layoff_history', 'employment_gap_months', 'relocation_willingness', 'status', 'status_encoded', 'internship_experience_encoded', 'career_switch_willingness_encoded', 'relevant_experience_encoded', 'job_role_match_encoded', 'bond_requirement_encoded', 'layoff_history_encoded', 'relocation_willingness_encoded', 'gender_Female', 'gender_Male', 'degree_specialization_Computer Science', 'degree_specialization_Electronics', 'degree_specialization_Information Technology', 'degree_specialization_Mechanical', 'degree_specialization_Others', 'company_tier_Tier 1', 'company_tier_Tier 2', 'company_tier_Tier 3', 'competiti

In [19]:
# ============================================================
# 📏 SCALING: StandardScaler on Numeric Columns
# ============================================================


scale_cols = [
    'age_years', 'ssc_percentage', 'hsc_percentage',
    'degree_percentage', 'technical_score', 'aptitude_score',
    'communication_score', 'skills_match_percentage',
    'certifications_count', 'years_of_experience',
    'previous_ctc_lpa', 'expected_ctc_lpa',
    'notice_period_days', 'employment_gap_months'
]

scaler = StandardScaler()                           # create the scaler tool

# fit_transform: learns the mean & std → then scales all values
df[scale_cols] = scaler.fit_transform(df[scale_cols])

print("\nAfter scaling — technical_score stats (should be near 0):")
print(df['technical_score'].describe().round(3))



After scaling — technical_score stats (should be near 0):
count    50000.000
mean         0.000
std          1.000
min         -2.365
25%         -0.683
50%         -0.003
75%          0.677
max          2.696
Name: technical_score, dtype: float64


In [20]:
# ============================================================
# ✅ FINAL CHECK
# ============================================================
print(f"\n{'='*50}")
print(f" Final dataset shape    : {df.shape}")
print(f" Missing values left    : {df.isnull().sum().sum()}")
print(f" Duplicate rows left    : {df.duplicated().sum()}")
print(f" Target (1=Placed)      : {df['status_encoded'].sum()}")
print(f" Target (0=Not Placed)  : {(df['status_encoded']==0).sum()}")
print(f"{'='*50}")

# Save the cleaned dataframe for next steps
df_clean = df.copy()                                # save as df_clean
print("\ndf_clean is ready for EDA and ML! 🎉")


 Final dataset shape    : (50000, 43)
 Missing values left    : 0
 Duplicate rows left    : 0
 Target (1=Placed)      : 15130
 Target (0=Not Placed)  : 34870

df_clean is ready for EDA and ML! 🎉


In [21]:
df.iloc[0]

age_years                                          -0.125697
ssc_percentage                                     -0.636961
hsc_percentage                                      1.517642
degree_percentage                                   0.266272
technical_score                                    -0.827689
aptitude_score                                      2.579495
communication_score                                -0.164031
skills_match_percentage                               0.4753
certifications_count                                0.505576
internship_experience                                     No
years_of_experience                                -0.358897
career_switch_willingness                            Willing
relevant_experience                                 Relevant
previous_ctc_lpa                                   -0.617938
expected_ctc_lpa                                    -0.70017
job_role_match                                   Not Matched
bond_requirement        

In [22]:
# Check the dtype is now int, not bool
print(df['gender_Male'].dtype)        # should print: int64 ✅
print(df['gender_Male'].unique())     # should print: [1 0] ✅

int64
[1 0]


In [23]:
df

,age_years,ssc_percentage,hsc_percentage,degree_percentage,technical_score,aptitude_score,communication_score,skills_match_percentage,certifications_count,internship_experience,...,degree_specialization_Electronics,degree_specialization_Information Technology,degree_specialization_Mechanical,degree_specialization_Others,company_tier_Tier 1,company_tier_Tier 2,company_tier_Tier 3,competition_level_High,competition_level_Low,competition_level_Medium
0,-0.125697,-0.636961,1.517642,0.266272,-0.827689,2.579495,-0.164031,0.475300,0.505576,No,...,0,0,0,0,0,0,1,0,0,1
1,-0.870819,-0.273841,-0.913011,-0.131413,0.328398,-0.948413,-0.507189,-0.052936,-0.338204,Yes,...,1,0,0,0,1,0,0,1,0,0
2,1.364547,0.498548,-0.415679,2.330301,0.372012,-0.545420,1.353205,0.129351,-0.338204,Yes,...,0,1,0,0,0,0,1,0,1,0
3,0.867799,0.531092,0.540261,0.227439,0.912740,-0.299800,-1.248310,-0.022399,0.505576,No,...,0,0,1,0,0,1,0,0,1,0
4,0.122677,-1.226614,-0.808280,0.977960,0.021268,0.173454,-0.470722,1.275857,-0.338204,No,...,0,1,0,0,0,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,0.371051,0.818788,1.371223,1.120342,0.330727,1.037620,-0.048967,-1.012198,-0.338204,Yes,...,1,0,0,0,0,1,0,0,0,1
49996,1.116173,1.287736,-0.419587,-0.694886,-1.226906,-1.230382,0.936891,0.324288,0.505576,No,...,0,0,1,0,0,0,1,1,0,0
49997,0.122677,-1.056179,-0.294918,0.830703,-0.632966,-2.420324,-1.136034,-1.626496,-0.338204,Yes,...,0,0,1,0,0,1,0,0,0,1
49998,1.612922,0.231018,0.615693,1.842846,-0.035659,-1.979153,0.383950,1.558785,1.349356,No,...,0,0,0,1,1,0,0,0,1,0


In [24]:
# Save the cleaned dataframe as a new CSV file
df.to_csv("HR_Job_Placement_Cleaned.csv", index=False)  
# index=False → don't save row numbers as a column
print("✅ Cleaned file saved!")

✅ Cleaned file saved!
